# Benchmark 1: throughput vs concurrency
Steady-state write throughput at concurrency 1/10/50/100, single-node baseline vs cluster. Run `bench/01-throughput-sweep.sh` first.

In [ ]:
import sys
sys.path.insert(0, ".")
import matplotlib.pyplot as plt
from plot_style import load, agg


In [ ]:
df = load("throughput-sweep.csv")
df

In [ ]:
summary = agg(df, ["clusterSize", "concurrency"], "reqPerSec")

fig, ax = plt.subplots()
for size, group in summary.groupby("clusterSize"):
    group = group.sort_values("concurrency")
    ax.errorbar(group["concurrency"], group["mean"], yerr=group["std"],
                marker="o", capsize=3, label=f"cluster size {size}")
ax.set_xscale("log")
ax.set_xlabel("concurrency")
ax.set_ylabel("requests/sec")
ax.set_title(f"Write throughput vs concurrency (mean ± stddev, n={df['run'].nunique()} runs)")
ax.legend()
plt.show()

### Plateau detection
The plateau (where added concurrency stops buying throughput) matters more than the peak - it's the point where replication/IO, not client load, is the bottleneck.

In [ ]:
for size, group in summary.groupby("clusterSize"):
    group = group.sort_values("concurrency").reset_index(drop=True)
    pct_change = group["mean"].pct_change() * 100
    plateau_row = group.iloc[1:][pct_change.iloc[1:].abs() < 5]
    plateau_c = plateau_row["concurrency"].iloc[0] if len(plateau_row) else None
    print(f"cluster size {size}: plateaus around concurrency={plateau_c}")